# Problem 2 — Transformer from scratch

The assignment forbids using `nn.MultiheadAttention` and `nn.Transformer*` in this problem. Implement scaled dot-product attention, multi-head attention, and the encoder block yourself in `from_scratch.py` before running dependent cells. Set the Transformer width from the instructor's missing individualisation table.

In [1]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'problem2').is_dir():
    if (ROOT / 'assignment' / 'problem2').is_dir():
        ROOT = ROOT / 'assignment'
    elif (ROOT.parent / 'problem2').is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from problem2.data import load_tiny_shakespeare, get_batch
from problem2.from_scratch import (
    scaled_dot_product_attention,
    MultiHeadAttention,
    EncoderBlock,
    SinusoidalPositionalEncoding,
    make_causal_mask,
)
from problem2.experiments import (
    CharacterTransformerLM,
    benchmark_context_lengths,
    plot_attention_heads,
    print_problem1_baseline,
    run_har_positional_comparison,
    run_language_ablations,
    train_character_model,
)

# Provisional values; replace those tied to the instructor's individualisation table.
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 4
CONTEXT = 128
SEEDS = [11, 21, 31]
HAR_NUM_LAYERS = 2
HAR_BATCH_SIZE = 128
HAR_EPOCHS = 10
LM_BATCH_SIZE = 64
LM_STEPS = 5000
LM_EVAL_INTERVAL = 250
LM_EVAL_BATCHES = 20
LM_ABLATION_STEPS = 1000
HAR_ROOT = ROOT / 'problem1/data/UCI HAR Dataset'
PROBLEM1_RESULTS_PATH = ROOT / 'problem1/results.csv'
RESULTS_PATH = ROOT / 'problem2/results.csv'
FIGURE_DIR = ROOT / 'problem2/figures'
SAMPLES_PATH = ROOT / 'problem2/generated_samples.txt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## Attention and encoder-block unit tests

Place assert checks directly after your implementations. Test row sums, agreement with `torch.nn.functional.scaled_dot_product_attention` to 1e-5, zero future weights under a causal mask, and the instructor's hand-computable 3-token example. Then check block output shape and parameter count.

In [2]:
import torch.nn.functional as F

q = torch.randn(2, 3, 5, 8, device=DEVICE)
k = torch.randn(2, 3, 5, 8, device=DEVICE)
v = torch.randn(2, 3, 5, 8, device=DEVICE)
attention_output, attention_weights = scaled_dot_product_attention(q, k, v)
reference_output = F.scaled_dot_product_attention(q, k, v)
assert torch.allclose(attention_output, reference_output, atol=1e-5, rtol=1e-5)
assert torch.allclose(
    attention_weights.sum(dim=-1),
    torch.ones_like(attention_weights.sum(dim=-1)),
    atol=1e-6,
)

causal_mask = make_causal_mask(5, device=DEVICE)
causal_output, causal_weights = scaled_dot_product_attention(q, k, v, causal_mask)
reference_causal = F.scaled_dot_product_attention(q, k, v, attn_mask=causal_mask)
assert torch.allclose(causal_output, reference_causal, atol=1e-5, rtol=1e-5)
assert torch.count_nonzero(causal_weights.masked_select(~causal_mask)) == 0

q_hand = torch.zeros(1, 1, 3, 2, device=DEVICE)
k_hand = torch.zeros_like(q_hand)
v_hand = torch.tensor([[[[1., 0.], [0., 2.], [3., 3.]]]], device=DEVICE)
hand_output, hand_weights = scaled_dot_product_attention(q_hand, k_hand, v_hand)
expected_output = torch.tensor([[[[4 / 3, 5 / 3]]]], device=DEVICE).expand_as(hand_output)
assert torch.allclose(hand_output, expected_output, atol=1e-6)
assert torch.allclose(hand_weights, torch.full_like(hand_weights, 1 / 3), atol=1e-6)

block = EncoderBlock(D_MODEL, NUM_HEADS, d_ff=4 * D_MODEL, dropout=0.0).to(DEVICE)
block_output, block_attention = block(
    torch.randn(2, 7, D_MODEL, device=DEVICE), return_attention=True
)
assert block_output.shape == (2, 7, D_MODEL)
weight_count = sum(parameter.numel() for parameter in block.parameters() if parameter.ndim > 1)
parameter_count = sum(parameter.numel() for parameter in block.parameters())
expected_weights = 4 * D_MODEL**2 + 2 * D_MODEL * (4 * D_MODEL)
assert weight_count == expected_weights
print('attention checks passed; 3-token output:', hand_output[0, 0, 0].tolist())
print('block weights:', weight_count, 'total including biases and LayerNorm:', parameter_count)

attention checks passed; 3-token output: [1.3333333730697632, 1.6666667461395264]
block weights: 196608 total including biases and LayerNorm: 198272


## HAR encoder classifier

Reuse Problem 1's subject-wise split and training-only normalization. Implement a CLS-token classifier using your encoder block. Compare no positional encoding, sinusoidal encoding, and learned positional encoding; record accuracy, parameter count, and training time.

In [3]:
# Loads Problem 1's subject-wise HAR split and compares all three position variants.
problem1_baseline = print_problem1_baseline(PROBLEM1_RESULTS_PATH)
har_encoder_records = run_har_positional_comparison(
    HAR_ROOT,
    SEEDS,
    D_MODEL,
    NUM_HEADS,
    HAR_NUM_LAYERS,
    HAR_BATCH_SIZE,
    HAR_EPOCHS,
    DEVICE,
    RESULTS_PATH,
)

Best Problem 1 baseline (GRU): accuracy=0.8782, params=14790, sec/epoch=4.956
HAR encoder none: accuracy=0.9183 +/- 0.0189, params=407046, sec/epoch=7.358 +/- 0.111
HAR encoder sinusoidal: accuracy=0.8987 +/- 0.0126, params=407046, sec/epoch=7.295 +/- 0.125
HAR encoder learned: accuracy=0.9139 +/- 0.0108, params=409222, sec/epoch=7.320 +/- 0.081


## Tiny Shakespeare causal language model

The supplied data helper downloads the text, checks the 65-character vocabulary, and makes the contiguous 90/10 split. Train with context 128, d=128, four heads and four layers; report validation nats and bits/character. Generate and save three labelled 500-character samples at temperatures 0.2, 0.8, and 1.5 using top-k sampling.

In [4]:
text, vocabulary, char_to_id, train_tokens, val_tokens = load_tiny_shakespeare()
id_to_char = {index: character for character, index in char_to_id.items()}
print('characters:', len(vocabulary), 'train tokens:', len(train_tokens), 'validation tokens:', len(val_tokens))
xb, yb = get_batch(train_tokens, batch_size=4, context=CONTEXT, device=DEVICE)
assert xb.shape == yb.shape == (4, CONTEXT)

lm_model, lm_metrics = train_character_model(
    train_tokens,
    val_tokens,
    vocab_size=len(vocabulary),
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    context=CONTEXT,
    batch_size=LM_BATCH_SIZE,
    steps=LM_STEPS,
    eval_interval=LM_EVAL_INTERVAL,
    eval_batches=LM_EVAL_BATCHES,
    seed=SEEDS[0],
    device=DEVICE,
    results_path=RESULTS_PATH,
    figure_dir=FIGURE_DIR,
    samples_path=SAMPLES_PATH,
    id_to_char=id_to_char,
    top_k=20,
)

import re
sentence_match = re.search(r'[^.!?]*[.!?]', text)
heatmap_sentence = sentence_match.group(0).strip() if sentence_match else text[:CONTEXT]
heatmap_sentence = heatmap_sentence[:CONTEXT]
sentence_ids = torch.tensor(
    [[char_to_id[character] for character in heatmap_sentence]],
    dtype=torch.long,
    device=DEVICE,
)
attention_figure = plot_attention_heads(
    lm_model, sentence_ids, FIGURE_DIR / 'language_model_attention_heads.png', heads=(0, 1)
)
print('attention heatmaps saved to:', attention_figure)

# These run the requested ablations and context benchmark; lower the step counts above for a pilot.
language_ablation_records = run_language_ablations(
    train_tokens,
    val_tokens,
    len(vocabulary),
    D_MODEL,
    NUM_LAYERS,
    CONTEXT,
    LM_BATCH_SIZE,
    LM_ABLATION_STEPS,
    LM_EVAL_INTERVAL,
    LM_EVAL_BATCHES,
    SEEDS,
    DEVICE,
    RESULTS_PATH,
    FIGURE_DIR,
    SAMPLES_PATH,
    id_to_char,
)
context_benchmark_records = benchmark_context_lengths(
    len(vocabulary),
    D_MODEL,
    NUM_HEADS,
    NUM_LAYERS,
    contexts=(64, 128, 256, 512),
    device=DEVICE,
    results_path=RESULTS_PATH,
    figure_dir=FIGURE_DIR,
)

characters: 65 train tokens: 1003854 validation tokens: 111540
step 1/5000: train=4.1533 nats, validation=4.1505 nats
step 250/5000: train=2.4309 nats, validation=2.4415 nats
step 500/5000: train=2.2101 nats, validation=2.2416 nats


KeyboardInterrupt: 